Давайте решим следующую задачу.<br>
Необходимо написать робота, который будет скачивать новости с сайта Лента.Ру и фильтровать их в зависимости от интересов пользователя. От пользователя требуется отмечать интересующие его новости, по которым система будет выделять области его интересов.<br>


Начнем с загрузки новостей. Для этого нам потребуется метод requests.get(url). Библиотека requests предоставляет серьезные возможности для загрузки информации из Интернет. Метод get получает URL стараницы и возвращает ее содержимое. В нашем случае результат будет получаться в формате html. <br>
Загрузим необходимые библиотеки.

In [1]:
import requests # Загрузка новостей с сайта.
from bs4 import BeautifulSoup # Превращалка html в текст.
import re # Регулярные выражения.

Теперь попробуем загрузить страницу новостей.

In [2]:
# Для пробы получаем первую страницу сайта.
requests.get("https://lenta.ru/")

<Response [200]>

Метод <i>requests.get()</i> возвращает объект Response, который содержит большое количество различной информации о загруженной (или незагруженной) странице. В краткой форме отображается только результат выполения запроса. В нашем случае это 200, нет ошибки.<br> 
Посмотрим что результат содержит еще.

In [3]:
resp = requests.get("https://lenta.ru/news/2018/08/24/clon/")

In [4]:
dir(resp)

['__attrs__',
 '__bool__',
 '__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__enter__',
 '__eq__',
 '__exit__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__iter__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__nonzero__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__setstate__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_content',
 '_content_consumed',
 '_next',
 'apparent_encoding',
 'close',
 'connection',
 'content',
 'cookies',
 'elapsed',
 'encoding',
 'headers',
 'history',
 'is_permanent_redirect',
 'is_redirect',
 'iter_content',
 'iter_lines',
 'json',
 'links',
 'next',
 'ok',
 'raise_for_status',
 'raw',
 'reason',
 'request',
 'status_code',
 'text',
 'url']

In [5]:
%%time
# %%time - Магия Jupyter - замеряет время выполнения ячейки. Должно быть первой строчкой в ячейке.
resp = requests.get("https://lenta.ru/news/2018/08/24/clon/")
print("cookies:", resp.cookies)
print("time to download:", resp.elapsed)
print("page encoding", resp.encoding)
print("Server response: ", resp.status_code)
print("Is everything ok? ", resp.ok)
print("Page's URL: ", resp.url)

cookies: <RequestsCookieJar[<Cookie lids=482DF5379FA4B6FF for .lenta.ru/>, <Cookie lenta_user_city=Moscow for .lenta.ru/>, <Cookie lid=vAsAAJN+zmk39UuVAe3yCAB= for .lenta.ru/>]>
time to download: 0:00:00.059482
page encoding utf-8
Server response:  200
Is everything ok?  True
Page's URL:  https://lenta.ru/news/2018/08/24/clon/
CPU times: user 33.9 ms, sys: 2.98 ms, total: 36.8 ms
Wall time: 62 ms


In [6]:
for c in resp.cookies:
    print(c)

<Cookie lids=482DF5379FA4B6FF for .lenta.ru/>
<Cookie lenta_user_city=Moscow for .lenta.ru/>
<Cookie lid=vAsAAJN+zmk39UuVAe3yCAB= for .lenta.ru/>


Но самое для нас интересное хранится в поле <i>text</i>, которое содержит собственно текст html-страницы.

In [7]:
#Берем первые 1000 символов новости.
resp.text[:1000]

'<!DOCTYPE html><html lang="ru"><head><title>В Сибири нашли подходящих для клонирования древних животных: Наука: Наука и техника: Lenta.ru</title><meta charset="utf-8" /><script>(function () {\n   try {\n      var PROCESS_COOKIE = "auth_in_process=1";\n      var REFERRER_LS_KEY = "saved_referrer";\n\n      function hasCookie() {\n        return document.cookie\n          .split("; ")\n          .indexOf(PROCESS_COOKIE) !== -1;\n      }\n\n      function setCookie() {\n        document.cookie = PROCESS_COOKIE + "; path=/"\n      }\n\n      var originalReferrer = document.referrer;\n\n      Object.defineProperty(document, \'referrer\', {\n        get: function() {\n          try {\n            const savedReferrer = localStorage.getItem(REFERRER_LS_KEY);\n            if (savedReferrer) return savedReferrer;\n            if (/^https?:\\/\\/id(-\\w+)?\\.sber\\.ru/i.test(originalReferrer)) return \'\';\n            return originalReferrer;\n          } catch (e) {\n            return origina

Количество служебной информации в странице явно превышает объем текста новости. У нас есть два пути: либо использовать библиотеку BeautyfulSoup для получения текста статьи, либо получить текст с использованием регулярных выражений.

Опробуем первый путь. Документация на библиотеку BeautyfulSoup находится <a href="https://www.crummy.com/software/BeautifulSoup/bs4/doc/">здесь</a>.

В ячейке ниже мы создаем объект BeautifulSoup, передаем в него текст html-страницы и сообщаем, что разбирать его надо при помощи библиотеки `html5lib`. Далее просим отдать текст страницы без html-тегов.

In [8]:
BeautifulSoup(resp.text, "html5lib").get_text()[:1000]

'В Сибири нашли подходящих для клонирования древних животных: Наука: Наука и техника: Lenta.ru(function () {\n   try {\n      var PROCESS_COOKIE = "auth_in_process=1";\n      var REFERRER_LS_KEY = "saved_referrer";\n\n      function hasCookie() {\n        return document.cookie\n          .split("; ")\n          .indexOf(PROCESS_COOKIE) !== -1;\n      }\n\n      function setCookie() {\n        document.cookie = PROCESS_COOKIE + "; path=/"\n      }\n\n      var originalReferrer = document.referrer;\n\n      Object.defineProperty(document, \'referrer\', {\n        get: function() {\n          try {\n            const savedReferrer = localStorage.getItem(REFERRER_LS_KEY);\n            if (savedReferrer) return savedReferrer;\n            if (/^https?:\\/\\/id(-\\w+)?\\.sber\\.ru/i.test(originalReferrer)) return \'\';\n            return originalReferrer;\n          } catch (e) {\n            return originalReferrer;\n          }\n        },\n      });\n\n      var params = new URLSearchPa

Да, убрать html-теги получилось. Но их содержимое осталось, в том числе и скрипты.<br>
Опробуем другой путь. Весь текст обычно оформляется тегом параграфа - &lt;p&gt;. Выберем весь текст из этих тегов. Заодно выберем и заголовок статьи, оформленный при помощи \<h1\>.

In [9]:
# Получили объект BeautifulSoup и скормили ему текст страницы.
bs = BeautifulSoup(resp.text, "html5lib") 
# Вот таким образом можно попросить отдать первый тег, отмеченный как h1. Вместо h1 можно написать любой другой тег.
title = bs.h1.text
# Получаем все параграфы (тег p), берем их текст без тегов и склеиваем в один текст.
text = "\n".join([p.text for p in bs.find_all("p")])
print(title, "\n-----\n", text)

В Сибири нашли подходящих для клонирования древних животных 
-----
 Реклама
Фото: AP
Российские палеонтологи обнаружили в Якутии тушу жеребенка, возраст которой достигает 30-40 тысяч лет, а также останки мамонта с мягкими тканями. Об этом сообщается в пресс-релизе на Phys.org.
Специалисты отмечают хорошее состояние тела лошади, пролежавшей в вечной мерзлоте. Таким образом, находка является потенциально пригодной для клонирования животного.
У найденного ископаемого, относящегося к вымершему виду Equus lenensis, сохранились кожа, шерсть, копыта, хвост и внутренние органы. Возраст жеребенка на момент смерти составлял примерно 2-3 месяца. Причиной смерти, вероятно, является попадание в какую-то «ловушку» естественного происхождения, поскольку видимых повреждений на теле не было.
У трупа были взяты образцы шерсти и биологических жидкостей для тщательного генетического анализа. По словам исследователей, на данный момент это самые хорошо сохранившиеся из всех останков древних лошадей.
В 2015 

Теперь напишем функцию, которая выгружает все новости за сутки. <br>
Обратим внимание, что для сайта Lenta.ru можно написать адрес в формате lenta.ru/ГГГГ/ММ/ДД/ (год, месяц, день) и получить все новости за этот день. Попробуем получить все адреса с такой страницы.

In [10]:
# Идем на страницу, получаем ее текст, отдаем в BeautifulSoup, ищем все теги ссылок - а.
BeautifulSoup(requests.get("http://lenta.ru/2018/08/25/").text, "html5lib").find_all("a")[:20]

[<a class="menu__nav-link _is-extra" href="/">Главное</a>,
 <a class="menu__nav-link _is-extra" href="/rubrics/russia/">Россия</a>,
 <a class="menu__nav-link _is-extra" href="/rubrics/world/">Мир</a>,
 <a class="menu__nav-link _is-extra" href="/rubrics/ussr/">Бывший СССР</a>,
 <a class="menu__nav-link _is-extra" href="/rubrics/economics/">Экономика</a>,
 <a class="menu__nav-link _is-extra" href="/rubrics/forces/">Силовые структуры</a>,
 <a class="menu__nav-link _is-extra" href="/rubrics/science/">Наука и техника</a>,
 <a class="menu__nav-link _is-extra" href="/rubrics/autolenta/">Авто</a>,
 <a class="menu__nav-link _is-extra" href="/rubrics/culture/">Культура</a>,
 <a class="menu__nav-link _is-extra" href="/rubrics/sport/">Спорт</a>,
 <a class="menu__nav-link _is-extra" href="/rubrics/media/">Интернет и СМИ</a>,
 <a class="menu__nav-link _is-extra" href="/rubrics/style/">Ценности </a>,
 <a class="menu__nav-link _is-extra" href="/rubrics/travel/">Путешествия</a>,
 <a class="menu__nav-li

Кажется, это опять немного не то, что нам нужно. Мы получили все ссылки, находящиеся на боковом меню, ссылки на события сегодняшнего дня и другие ненужные нам вещи. <br>
Смотрим в содержимое html-страницы и обращаем внимание, что все интересные нам ссылки оформлены как заголовки третьего уровня - &lt;h3&gt;. Извлечем все такие фрагменты, а потом извлечем собственно адреса, помеченные атрибутом href тега &lt;a&gt;.

In [11]:
# Теперь выделим только то, что взято в тег h3.
beau = BeautifulSoup(requests.get("http://lenta.ru/2018/08/25/").text, "html5lib")
tags = beau.find_all("a", attrs={'class': 'card-full-news'})
# Формируем список ссылок. Для этого берем первую (кстати, единственную) ссылку из каждого выделенного
# фрагмента, у нее берем значение параметра href. Так как ссылки внутренние, добавляем к ним адрес сайта.
links = ["https://lenta.ru"+l["href"] for l in tags]
print(links)

['https://lenta.ru/news/2018/08/25/potomu/', 'https://lenta.ru/news/2018/08/25/rastvorova/', 'https://lenta.ru/news/2018/08/25/razvedka/', 'https://lenta.ru/news/2018/08/25/poezd/', 'https://lenta.ru/news/2018/08/25/bolton/', 'https://lenta.ru/news/2018/08/25/nur/', 'https://lenta.ru/news/2018/08/25/firetornado/', 'https://lenta.ru/news/2018/08/25/drake/', 'https://lenta.ru/news/2018/08/25/merkel/', 'https://lenta.ru/news/2018/08/25/serena/', 'https://lenta.ru/news/2018/08/25/python/', 'https://lenta.ru/news/2018/08/25/ukrain/', 'https://lenta.ru/news/2018/08/25/vsetaki_pustili/', 'https://lenta.ru/news/2018/08/25/boycott_curtis/', 'https://lenta.ru/news/2018/08/25/twitterrr/', 'https://lenta.ru/news/2018/08/25/milliardy/', 'https://lenta.ru/news/2018/08/25/otrajenie/', 'https://lenta.ru/news/2018/08/25/dope_ioc/', 'https://lenta.ru/news/2018/08/25/edro/', 'https://lenta.ru/news/2018/08/25/galaktika_v_opasnosti/', 'https://lenta.ru/news/2018/08/25/seady_steady/', 'https://lenta.ru/news

Если теперь написать функцию, которая будет перебирать все адреса и получать из них тексты новостей, то мы получим все новости за определенные сутки. Но это мы сделаем чть позже, а пока просто оформим код загрузки статьи в виде функции.

In [13]:
# Загрузка статьи по URL.
def getOneLentaArticle(url):
    """ getLentaArticle gets the body of an article from Lenta.ru"""
    # Получает текст страницы.
    resp=requests.get(url)
    # Загружаем текст в объект типа BeautifulSoup.
    bs=BeautifulSoup(resp.text, "html5lib") 
    # Получаем заголовок статьи.
    aTitle=bs.h1.text.replace("\xa0", " ")
    # Получаем текст статьи.
    anArticle=BeautifulSoup(" ".join([p.text for p in bs.find_all("p")]), "html5lib").get_text().replace("\xa0", " ")
    # Возвращаем кортеж из заголовка и текста статьи.
    return aTitle, anArticle


### XPath
[Ссылка 1](https://habr.com/ru/post/526774/)

[Ссылка 2](https://habr.com/ru/post/464897/)

XPath позволяет задать шаблон для пути от корня XML-дерева к интересующей нас вершине.

- . - корень XML-дерева
- / - переход на один уровень ниже.
- // - переход на ноль или больше уровней вниз.
- \* - любая вершина.
- xyz - название вершины.
- [@feature] - вершина с параметром feature.
- [@feature='111'] - вершина с параметром feature, равным "111".
- xyz[n] - n-ый потомок вершины xyz.

А теперь давайте посмотрим как мы можем при помощи XPath обрабатывать HTML-документы.

https://lenta.ru/news/2021/02/27/apple_effect/

In [14]:
from lxml import html

In [18]:
page = requests.get('https://lenta.ru/news/2021/02/27/apple_effect/')

In [21]:
tree = html.fromstring(page.text)
print(tree.xpath(".//h1")[0].text_content())
print(tree.xpath(".//a[contains(@class, 'topic-header__time')]")[0].text_content().strip())
print(tree.xpath(".//div[contains(@class, 'topic-authors')]")[0].text_content().strip(), '\n')
print(tree.xpath(".//meta[@name='og:description']")[0].get("content"), '\n')

for p in tree.xpath(".//div[contains(@class, 'topic-body')]/p"):
    print(p.text_content())

Обнаружен неожиданный эффект от употребления яблок
15:35, 27 февраля 2021
Соня Кошечкина (Редактор) 

Ученые из Университета Квинсленда и Немецкого центра нейродегенеративных заболеваний  обнаружили неожиданный эффект от употребления яблок. Опыты проводились на мышах. Специалисты культивировали стволовые клетки мозга взрослых мышей и добавляли в них содержащиеся в яблоках фитонутриенты. 

Ученые из Университета Квинсленда и Немецкого центра нейродегенеративных заболеваний обнаружили неожиданный эффект от употребления яблок. Результаты исследования появились в научном журнале Stem Cell Reports.
Опыты проводились на мышах. Специалисты культивировали стволовые клетки мозга взрослых мышей и добавляли в них содержащиеся в яблоках фитонутриенты. Исследование показало, что высокая концентрация фитонутриентов способствует образованию новых нейронов.
По словам ученых, определенные фитонутриенты положительно влияют на работу органов, в том числе мозга. Выяснилось, что они оказывают на организм т

А теперь та же страница, но через BeautyfulSoup.

In [23]:
souped = BeautifulSoup(page.text)

title = souped("h1")[0].get_text()
when = souped.find_all("a", attrs={'class': 'topic-header__time'})[0].get_text().strip()
author = souped.find_all("div", attrs={'class': 'topic-authors'})[0].get_text()
description = souped.find_all("meta", attrs={'name': 'og:description'})[0]["content"]
print(title)

for p in souped.find_all("div", attrs={'class': 'topic-body'})[0]("p"): 
    print(p.get_text())
    


Обнаружен неожиданный эффект от употребления яблок
Анатолий Жданов / «Коммерсантъ»
Ученые из Университета Квинсленда и Немецкого центра нейродегенеративных заболеваний обнаружили неожиданный эффект от употребления яблок. Результаты исследования появились в научном журнале Stem Cell Reports.
Опыты проводились на мышах. Специалисты культивировали стволовые клетки мозга взрослых мышей и добавляли в них содержащиеся в яблоках фитонутриенты. Исследование показало, что высокая концентрация фитонутриентов способствует образованию новых нейронов.
По словам ученых, определенные фитонутриенты положительно влияют на работу органов, в том числе мозга. Выяснилось, что они оказывают на организм тот же эффект, что и физическая активность, которая также стимулирует нейрогенез.
Ранее ученые из Технологического университета австрийского Граца выяснили, что большинство людей неправильно едят яблоки. Исследователи утверждают, что до 90 процентов полезных веществ сосредоточены в сердцевине этого фрукта, и 

In [24]:
title, when, author

('Обнаружен неожиданный эффект от употребления яблок',
 '15:35, 27 февраля 2021',
 'Соня Кошечкина (Редактор)')

### Cookies

Библиотека также позволяет работать с куки. 

In [25]:
le = requests.get("https://lenta.ru")
print(le.cookies)

<RequestsCookieJar[<Cookie lids=4823D7B89FE98252 for .lenta.ru/>, <Cookie lenta_user_city=Moscow for .lenta.ru/>, <Cookie lid=vAsAAKR/zmm410YCAbR+CAB= for .lenta.ru/>]>


In [26]:
c1 = le.cookies
# c1
c1.set('sdjkfsa', '123445') 
# c1

Cookie(version=0, name='sdjkfsa', value='123445', port=None, port_specified=False, domain='', domain_specified=False, domain_initial_dot=False, path='/', path_specified=True, secure=False, expires=None, discard=True, comment=None, comment_url=None, rest={'HttpOnly': None}, rfc2109=False)

In [27]:
coo = requests.cookies.RequestsCookieJar()
coo.set("asfdasfd", "12341234")
res = requests.get("https://lenta.ru", cookies=coo)

### Заголовки
При работе с http, мы обмениваемся с сервером заголовками. Например, сервер рассказывает свою версию, какого вида данные он нам вернул, какие он поддерживает протоколы.

In [34]:
res.headers

{'Server': 'nginx', 'Date': 'Thu, 02 Apr 2026 14:39:36 GMT', 'Content-Type': 'text/html; charset=utf-8', 'Transfer-Encoding': 'chunked', 'Connection': 'keep-alive', 'Keep-Alive': 'timeout=50', 'X-XSS-Protection': '0', 'X-Content-Type-Options': 'nosniff', 'X-Permitted-Cross-Domain-Policies': 'none', 'Referrer-Policy': 'strict-origin-when-cross-origin', 'P3P': 'CP="This Is Potato!", CP="NON DSP NID ADMa DEVa TAIa PSAa PSDa OUR IND UNI COM NAV"', 'Link': '<https://icdn.lenta.ru/assets/webpack/indexPageOwl.608da3e8.css>; rel=preload; as=style; nopush,<https://icdn.lenta.ru/assets/webpack/owlBundle.7588bd84.css>; rel=preload; as=style; nopush,<//ssp.rambler.ru/capirs_async.js>; rel=preload; as=script; nopush', 'Vary': 'Accept', 'ETag': 'W/"23d38362cc6436e3f75942c574f31bf6"', 'Cache-Control': 'max-age=0, private, must-revalidate', 'X-Request-Id': 'f4177a78-4276-447d-a157-1f3e8191edc6', 'X-Runtime': '0.905696', 'Set-Cookie': 'lids=4827D7CE9FEA039D;path=/;Max-Age=1800;domain=.lenta.ru, lenta_u

Мы тоже можем ему передать заголовки с интересной для нас информацией (можно посмотреть в отладочной информации браузера). Например, наша программа может начать представляться другим браузером.

In [47]:
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; WOW64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/85.0.4183.102 YaBrowser/20.9.3.136 Yowser/2.5 Safari/537.36'}

response = requests.get('https://lenta.ru', headers=headers)

### Составной пример с кукис и заголовками

In [30]:
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; WOW64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/85.0.4183.102 YaBrowser/20.9.3.136 Yowser/2.5 Safari/537.36'}
coo = requests.cookies.RequestsCookieJar()

# ww = requests.get("https://warszawa.wyborcza.pl/warszawa/7,54420,31835524,tak-metro-zmienilo-warszawe-pokazujemy-zdjecia-przed-i-po-budowie.html?squid_js=false",
ww = requests.get("https://warszawa.wyborcza.pl/warszawa/7,54420,31835524,tak-metro-zmienilo-warszawe-pokazujemy-zdjecia-przed-i-po-budowie.html",
                  headers=headers,
                  cookies=coo
                 )
souped = BeautifulSoup(ww.text)

souped("div")

[<div class="msg-container">
 <div id="message">
 <img src="https://bi.gazeta.pl/im/3/17117/m17117193.png"/>
 <div id="info-adblock">
 <h1>Wyłącz AdBlocka/uBlocka</h1>
 <p class="head">Aby czytać nasze artykuły wyłącz AdBlocka/uBlocka lub dodaj wyjątek dla naszej domeny.</p>
 <p class="lead">Spokojnie, dodanie wyjątku nie wyłączy blokowania reklam.</p>
 </div>
 <div id="info-ups">
 <h1>Ups!</h1>
 <p class="lead">Nieznany błąd - nie można wyświetlić strony</p>
 </div>
 </div>
 <div class="advertHolder adHolder" id="adHolder" style="display: block"></div>
 </div>,
 <div id="message">
 <img src="https://bi.gazeta.pl/im/3/17117/m17117193.png"/>
 <div id="info-adblock">
 <h1>Wyłącz AdBlocka/uBlocka</h1>
 <p class="head">Aby czytać nasze artykuły wyłącz AdBlocka/uBlocka lub dodaj wyjątek dla naszej domeny.</p>
 <p class="lead">Spokojnie, dodanie wyjątku nie wyłączy blokowania reklam.</p>
 </div>
 <div id="info-ups">
 <h1>Ups!</h1>
 <p class="lead">Nieznany błąd - nie można wyświetlić strony<

In [31]:
ccc = ww.cookies

In [32]:
ccc

<RequestsCookieJar[Cookie(version=0, name='SsoSessionPermanent', value='68249473a0c3710286b3103ec700bc8914f42059ff032264ae4dd132b6293bab', port=None, port_specified=False, domain='.wyborcza.pl', domain_specified=True, domain_initial_dot=False, path='/', path_specified=True, secure=True, expires=1932820816, discard=False, comment=None, comment_url=None, rest={'SameSite': 'None'}, rfc2109=False), Cookie(version=0, name='SquidLocalUID', value='55dd8b418f8038e22cf0dc9c', port=None, port_specified=False, domain='.wyborcza.pl', domain_specified=True, domain_initial_dot=False, path='/', path_specified=True, secure=True, expires=1869748816, discard=False, comment=None, comment_url=None, rest={'SameSite': 'None'}, rfc2109=False), Cookie(version=0, name='GW_SID', value='3F3AF0A01168D17529771E41924B5DCD.tomwybo90', port=None, port_specified=False, domain='warszawa.wyborcza.pl', domain_specified=False, domain_initial_dot=False, path='/', path_specified=True, secure=True, expires=None, discard=True

In [33]:
ww.text

'\n<!DOCTYPE html>\n<html>\n<head>\n    <meta charset="utf-8"/>\n    <title>Wyborcza.pl</title>\n    <noscript>\n        <meta http-equiv="Refresh" content="0; URL=https://warszawa.wyborcza.pl/warszawa/7,54420,31835524,tak-metro-zmienilo-warszawe-pokazujemy-zdjecia-przed-i-po-budowie.html?squid_js=false"/>\n    </noscript>\n    <meta http-equiv="cache-control" content="max-age=0"/>\n    <meta http-equiv="cache-control" content="no-cache"/>\n    <meta http-equiv="expires" content="0"/>\n    <meta http-equiv="expires" content="Tue, 01 Jan 1980 1:00:00 GMT"/>\n    <meta http-equiv="pragma" content="no-cache"/>\n    <link rel="shortcut icon" href="https://static.im-g.pl/aliasy/foto/wyborcza/favicon.ico">\n    <style type="text/css">\n        body {\n            font-family: Arial, sans-serif;\n            font-size: 13px;\n        }\n\n        h1 {\n            font-size: 16px;\n        }\n\n        a {\n            color: #146cb4;\n            text-decoration: none;\n        }\n\n        

### Сессия

Если вам необходимо ввести пароль, а потом работать как зарегистрированный пользователь, лучше использовать сессию, которая запомнит все данные. Об этом можно почитать [здесь](https://requests.readthedocs.io/en/latest/user/advanced/).

### Работа с бинарными файлами

Попробуем загрузить pdf-файл со статьей с сайта Киберленинка.

Правда, если мы будем загружать его все вместе с одного IP-адреса, то нас могут забанить за массовое скачивание.

In [37]:
# pdf1 = requests.get("https://cyberleninka.ru/article/n/semantiko-sintaksicheskoe-opisanie-glagolov-professionalnoy-deyatelnosti-iz-materialov-k-semantiko-grammaticheskomu-slovaryu/pdf")
pdf1 = requests.get("https://journals.rcsi.science/0023-4206/article/download/276519/255461")


Можно заметить, что у объекта есть несколько полей, хранящих бинарные строки, например, `content` и `raw`. Это, собственно, содержимое загруженного файла.

Еще у него есть свойства на случай, если был загружен json (`json`) или текст (мы его уже использовали - `text`).

In [38]:
dir(pdf1)

['__attrs__',
 '__bool__',
 '__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__enter__',
 '__eq__',
 '__exit__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__iter__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__nonzero__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__setstate__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_content',
 '_content_consumed',
 '_next',
 'apparent_encoding',
 'close',
 'connection',
 'content',
 'cookies',
 'elapsed',
 'encoding',
 'headers',
 'history',
 'is_permanent_redirect',
 'is_redirect',
 'iter_content',
 'iter_lines',
 'json',
 'links',
 'next',
 'ok',
 'raise_for_status',
 'raw',
 'reason',
 'request',
 'status_code',
 'text',
 'url']

Взглянем на содержимое загруженного файла. В первых байтах виден заголовок pdf-файла с автором, использованной программой и другими данными.

In [39]:
pdf1.content[:200]

b'%PDF-1.7\r%\xe2\xe3\xcf\xd3\r\n206 0 obj\r<</Linearized 1/L 1566561/O 209/E 55255/N 12/T 1562320/H [ 996 1104]>>\rendobj\r          \r\nxref\r\n206 35\r\n0000000016 00000 n\r\n0000002100 00000 n\r\n0000002283 00000 n\r\n0000002319'

Поставим себе библиотеку для чтения pdf-файлов и посмотрим на содержимое первой страницы.

In [41]:
!pip install --break-system-packages pypdf

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.7/333.7 kB 2.2 MB/s eta 0:00:00 MB/s eta 0:00:01


In [42]:
from pypdf import PdfReader
import io

Библиотека принимает на вход открытый файл, но при помощи `io.BytesIO()` мы сможем имитировать чтение из файла, которое будет брать данные из памяти.

In [43]:
f = io.BytesIO(pdf1.content)

reader = PdfReader(f)

Итак, посмотрим на текстовое содержимое первой страницы.

In [44]:
number_of_pages = len(reader.pages)
page = reader.pages[0]
text = page.extract_text()

In [45]:
text

'КОСМИЧЕСКИЕ ИССЛЕДОВАНИЯ,  2024, том 62,  № 5, с. 444–455\n444\n      \nУДК: 629.78 : 531\nМЕТОД ПОСТРОЕНИЯ НИЗКОЭНЕРГЕТИЧЕСКИХ ТРАЕКТОРИЙ \nВЫВЕДЕНИЯ КОСМИЧЕСКОГО АППАРАТА НА ОРБИТЫ \nИСКУССТВЕННОГО СПУТНИКА ЛУНЫ\n© 2024 г. С. А. Бобер 1,2, С. А. Аксенов 1,2,*\n1Национальный исследовательский институт «Высшая школа экономики», Москва, Россия\n2Институт космических исследований РАН, Москва, Россия\n*saksenov@hse.ru\nПоступила в редакцию 25.09.2023\nПосле доработки 13.02.2024\nПринята к публикации 15.02.2024\nВ работе предлагается метод построения траекторий выведения космического аппарата на \nкруговую полярную орбиту искусственного спутника Луны (ИСЛ), основанный на использовании \nсвойств инвариантных многообразий решений круговой ограниченной задачи трех тел. Такой \nподход по сравнению с классическим гомановским переходом позволяет существенно сократить \nтормозной импульс за счет увеличения времени перелета. Процесс построения орбит перелета \nвключает два этапа. На первом этапе 

М-да... С разбивкой на строки и абзацы полная беда. Попробуем хотя бы устранить преносы.

In [46]:
text.replace(' -\n', '').replace('-\n', '').replace('\n', ' ')

'КОСМИЧЕСКИЕ ИССЛЕДОВАНИЯ,  2024, том 62,  № 5, с. 444–455 444        УДК: 629.78 : 531 МЕТОД ПОСТРОЕНИЯ НИЗКОЭНЕРГЕТИЧЕСКИХ ТРАЕКТОРИЙ  ВЫВЕДЕНИЯ КОСМИЧЕСКОГО АППАРАТА НА ОРБИТЫ  ИСКУССТВЕННОГО СПУТНИКА ЛУНЫ © 2024 г. С. А. Бобер 1,2, С. А. Аксенов 1,2,* 1Национальный исследовательский институт «Высшая школа экономики», Москва, Россия 2Институт космических исследований РАН, Москва, Россия *saksenov@hse.ru Поступила в редакцию 25.09.2023 После доработки 13.02.2024 Принята к публикации 15.02.2024 В работе предлагается метод построения траекторий выведения космического аппарата на  круговую полярную орбиту искусственного спутника Луны (ИСЛ), основанный на использовании  свойств инвариантных многообразий решений круговой ограниченной задачи трех тел. Такой  подход по сравнению с классическим гомановским переходом позволяет существенно сократить  тормозной импульс за счет увеличения времени перелета. Процесс построения орбит перелета  включает два этапа. На первом этапе производится анализ

### Потоковое аудио
Теперь попробуем работать с потоковым аудио. Я начал со страницы http://radio.garden , которая хранит ссылки на интернет-радиостанции. С ее помощью я вышел вот на этот адрес. Попробуем грузить с него потоковое аудио в файл.

In [47]:
stream_url = 'https://radiorecord.hostingradio.ru/198096.aacp?listening-from-radio-garden=1680922353'
r = requests.get(stream_url, stream=True)

with open('stream.mp3', 'wb') as f:
    try:
        for block in r.iter_content(1024):
            f.write(block)
    except KeyboardInterrupt:
        pass

А попробуем теперь прослушать это аудио.

In [49]:
!pip install --break-system-packages pydub

Defaulting to user installation because normal site-packages is not writeable


In [50]:
from pydub import AudioSegment
from pydub.playback import play

/home/edward/.local/lib/python3.12/site-packages/pydub/utils.py:170: RuntimeWarning: Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work
  warn("Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work", RuntimeWarning)


In [51]:
import io

In [53]:
stream_url = 'http://stream1.waszeradiofm.pl:8000/;?listening-from-radio-garden=1696354810'
r = requests.get(stream_url, stream=True)

try:
    for block in r.iter_content(42400):
        f = io.BytesIO(block)
        sound = AudioSegment.from_file(f)
        play(sound)
except KeyboardInterrupt:
    pass

Input #0, wav, from '/tmp/tmpv2x651lz.wav':   0KB sq=    0B f=0/0   
  Duration: 00:00:02.66, bitrate: 1411 kb/s
  Stream #0:0: Audio: pcm_s16le ([1][0][0][0] / 0x0001), 44100 Hz, 2 channels, s16, 1411 kb/s
   2.60 M-A:  0.000 fd=   0 aq=    0KB vq=    0KB sq=    0B f=0/0   

Input #0, wav, from '/tmp/tmpabp8sbq7.wav':   0KB sq=    0B f=0/0   
  Duration: 00:00:02.64, bitrate: 1411 kb/s
  Stream #0:0: Audio: pcm_s16le ([1][0][0][0] / 0x0001), 44100 Hz, 2 channels, s16, 1411 kb/s
   2.53 M-A: -0.000 fd=   0 aq=    0KB vq=    0KB sq=    0B f=0/0   

Input #0, wav, from '/tmp/tmpe54c_992.wav':   0KB sq=    0B f=0/0   
  Duration: 00:00:02.66, bitrate: 1411 kb/s
  Stream #0:0: Audio: pcm_s16le ([1][0][0][0] / 0x0001), 44100 Hz, 2 channels, s16, 1411 kb/s
   2.59 M-A:  0.000 fd=   0 aq=    0KB vq=    0KB sq=    0B f=0/0   

Input #0, wav, from '/tmp/tmp4ymvt7vu.wav':   0KB sq=    0B f=0/0   
  Duration: 00:00:02.64, bitrate: 1411 kb/s
  Stream #0:0: Audio: pcm_s16le ([1][0][0][0] / 0x0001), 44100 Hz, 2 channels, s16, 1411 kb/s
   0.35 M-A:  0.000 fd=   0 aq=  176KB vq=    0KB sq=    0B f=0/0   